Cấu hình — chỉ xử lý dữ liệu offline; không publish.

In [ ]:
from pathlib import Path
import sys, platform, json
import pandas as pd
here = Path.cwd().resolve()
DATA_PREPARATION_DIR = next((p if p.name == 'data-preparation' else p / 'data-preparation' for p in [here, *here.parents] if (p if p.name == 'data-preparation' else p / 'data-preparation').is_dir()), None)
if DATA_PREPARATION_DIR is None:
    raise RuntimeError('Mở kernel trong repo hoặc đặt DATA_PREPARATION_DIR')
sys.path.insert(0, str(DATA_PREPARATION_DIR / 'scripts'))
import data_contract as dc
from prepare_contract_templates import example_bundle
SNAPSHOTS = DATA_PREPARATION_DIR / 'snapshots'
REPORTS_DIR = DATA_PREPARATION_DIR / 'datasets/data-contract/reports'
print(platform.python_version(), pd.__version__, SNAPSHOTS)


Dictionary — xem toàn bộ trường, kiểu và giá trị cho phép.

In [ ]:
for entity, definition in dc.ENTITIES.items():
    print(entity, 'primaryKey=', definition['primaryKey'])
    display(pd.DataFrame([dict(field=name, **field) for name, field in definition['fields'].items()]))


Đọc JSON/CSV ví dụ — fixture tổng hợp; không phải quán thật.

In [ ]:
bundle = json.loads((DATA_PREPARATION_DIR / 'templates/examples/dataset.json').read_text())
assert bundle == dc.read_csv_bundle(DATA_PREPARATION_DIR / 'templates/examples')
print({key:len(rows) for key,rows in bundle['entities'].items()})
for entity in ['dishes','venues','venueDishes','weeklySchedules','dateExceptions']:
    print(entity)
    display(pd.DataFrame(bundle['entities'][entity]))


Kiểm contract — lỗi ghi entity, hàng và trường.

In [ ]:
errors = dc.validate(bundle)
assert not errors, errors
print('Fixture valid; publicationPerformed=False')
import copy
invalid = copy.deepcopy(bundle)
invalid['entities']['venueDishes'][0]['priceMin'] = -1
display(pd.DataFrame(dc.validate(invalid)))


Chuyển đổi — xuất bản sao vào datasets; không ghi đè templates.

In [ ]:
OUTPUT_DIR = DATA_PREPARATION_DIR / 'datasets/data-contract/processed/example_csv'
dc.write_csv_bundle(bundle, OUTPUT_DIR)
readback = dc.read_csv_bundle(OUTPUT_DIR)
assert readback == bundle and not dc.validate(readback)
print(OUTPUT_DIR)
stats = {'entities': len(bundle['entities']), 'rows':sum(len(rows) for rows in bundle['entities'].values()), 'apiCalls':0, 'fixtureOnly': True, 'publicationPerformed':False}
print(stats)
